# VIVE — Intent & Behaviour Classifier Training (Colab / Kaggle)

Trains the VIVE text classifiers on cloud GPU with a **larger backbone** than the local 4 GB card can hold.

**Rules this notebook follows**
- Never fabricate a metric. Every number printed comes from an executed run.
- The evaluation split keeps its natural class distribution, so reported F1 describes the real, imbalanced problem.
- Results are valid only for the corpus named in the manifest. They are **not** a VIVE end-to-end accuracy claim and **not** a call-transcript benchmark.

**Restartable:** a checkpoint is written every epoch to `OUTPUT_DIR`.

In [ ]:
# GPU check. Do NOT assume one is allocated.
import subprocess
try:
    print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv']).decode())
except Exception as e:
    print('No GPU detected:', e)
    print('This notebook runs on CPU but will be very slow. Prefer a GPU runtime.')

In [ ]:
!pip -q install "transformers>=4.44,<5" "scikit-learn>=1.5" "accelerate>=0.33"

## 1. Data

Upload `data/manifests/scamshield.{train,val,test}.jsonl` from the VIVE repo.

The manifests are the source of truth for splits and labels. **Do not re-split here** — the splits are content-addressed to prevent duplicate leakage, and re-splitting destroys that guarantee.

In [ ]:
import os, json

MANIFEST_DIR = 'manifests'
os.makedirs(MANIFEST_DIR, exist_ok=True)

if not os.path.exists(MANIFEST_DIR + '/scamshield.train.jsonl'):
    from google.colab import files   # on Kaggle, point MANIFEST_DIR at /kaggle/input instead
    print('Upload scamshield.train.jsonl, scamshield.val.jsonl, scamshield.test.jsonl')
    for name in files.upload():
        os.rename(name, MANIFEST_DIR + '/' + name)

def load(split):
    with open(MANIFEST_DIR + '/scamshield.' + split + '.jsonl', encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

train, val, test = load('train'), load('val'), load('test')
print('train=%d val=%d test=%d' % (len(train), len(val), len(test)))

## 2. Configuration

`MODEL_NAME` is the main reason to run in the cloud. MuRIL is the better Indic backbone but needs torch >= 2.6 — its checkpoint is a `.bin`, and transformers refuses `torch.load` on older torch (CVE-2025-32434). Colab ships a recent torch, so MuRIL is usable here where it was not locally.

In [ ]:
TASK       = 'intent'                      # 'intent' or 'behavior'
MODEL_NAME = 'google/muril-base-cased'     # or 'FacebookAI/xlm-roberta-base'
EPOCHS     = 3
BATCH_SIZE = 32
MAX_LENGTH = 128
LR         = 3e-5
SEED       = 20260921
OUTPUT_DIR = '/content/vive-' + TASK + '-classifier'

INTENT_LABELS = ['NORMAL_CONVERSATION','OTP_REQUEST','PASSWORD_REQUEST',
    'CARD_DETAILS_REQUEST','BANKING_CREDENTIAL_REQUEST','MONEY_TRANSFER_REQUEST',
    'ACCOUNT_CHANGE_REQUEST','REMOTE_ACCESS_REQUEST','URGENT_ACTION',
    'THREAT_OR_INTIMIDATION','CONFIDENTIAL_INFORMATION','UNKNOWN']
BEHAVIOR_LABELS = ['AUTHORITY_IMPERSONATION','URGENCY','THREAT','FEAR',
    'SECRECY','PRESSURE','REWARD_PROMISE','NORMAL']
LABELS = INTENT_LABELS if TASK == 'intent' else BEHAVIOR_LABELS
print(TASK, ':', len(LABELS), 'labels')

In [ ]:
import numpy as np, torch, random
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, f1_score

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
idx = {l: i for i, l in enumerate(LABELS)}

class DS(Dataset):
    def __init__(self, rows, tok): self.rows, self.tok = rows, tok
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        enc = self.tok(r['text'], truncation=True, max_length=MAX_LENGTH,
                       padding='max_length', return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if TASK == 'intent':
            item['labels'] = torch.tensor(idx[r['label_intent']])
        else:
            y = torch.zeros(len(LABELS))
            for b in r['label_behaviors']: y[idx[b]] = 1.0
            item['labels'] = y
        return item

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS),
    problem_type='single_label_classification' if TASK == 'intent' else 'multi_label_classification',
).to(device)
print('%.1fM params on %s' % (sum(p.numel() for p in model.parameters())/1e6, device))

In [ ]:
# Class weights: inverse frequency, capped. A class with zero support gets
# weight 0 so it cannot absorb gradient from nothing.
counts = np.zeros(len(LABELS))
for r in train:
    if TASK == 'intent': counts[idx[r['label_intent']]] += 1
    else:
        for b in r['label_behaviors']: counts[idx[b]] += 1

w = np.zeros_like(counts); present = counts > 0
w[present] = counts[present].sum() / (present.sum() * counts[present])
w = torch.tensor(np.clip(w, 0, 20), dtype=torch.float, device=device)

for l, c, wt in zip(LABELS, counts, w.cpu().numpy()):
    print('  %-28s n=%7d  weight=%.2f' % (l, int(c), wt))

In [ ]:
loss_fn = (torch.nn.CrossEntropyLoss(weight=w) if TASK == 'intent'
           else torch.nn.BCEWithLogitsLoss(pos_weight=w))
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

train_dl = DataLoader(DS(train, tok), batch_size=BATCH_SIZE, shuffle=True)
val_dl   = DataLoader(DS(val, tok),   batch_size=BATCH_SIZE*2)
test_dl  = DataLoader(DS(test, tok),  batch_size=BATCH_SIZE*2)

steps = len(train_dl) * EPOCHS
sched = get_linear_schedule_with_warmup(opt, int(steps*0.06), steps)
scaler = torch.amp.GradScaler('cuda', enabled=device == 'cuda')

@torch.no_grad()
def evaluate(dl):
    model.eval(); L = []; Y = []
    for b in dl:
        y = b.pop('labels'); b = {k: v.to(device) for k, v in b.items()}
        with torch.autocast('cuda', torch.float16, enabled=device == 'cuda'):
            L.append(model(**b).logits.float().cpu())
        Y.append(y)
    L = torch.cat(L); Y = torch.cat(Y)
    if TASK == 'intent':
        p = L.argmax(-1).numpy(); t = Y.numpy()
    else:
        p = (torch.sigmoid(L) >= 0.5).int().numpy(); t = Y.int().numpy()
    return f1_score(t, p, average='macro', zero_division=0), t, p

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

for epoch in range(EPOCHS):
    model.train(); total = 0.0
    for step, b in enumerate(train_dl):
        y = b.pop('labels').to(device); b = {k: v.to(device) for k, v in b.items()}
        with torch.autocast('cuda', torch.float16, enabled=device == 'cuda'):
            loss = loss_fn(model(**b).logits, y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        total += loss.item()
        if (step+1) % 200 == 0:
            print('  e%d %d/%d loss=%.4f' % (epoch+1, step+1, len(train_dl), total/(step+1)), flush=True)
    f1, _, _ = evaluate(val_dl)
    print('epoch %d: val macro_f1=%.4f' % (epoch+1, f1))
    # Checkpoint every epoch so the run is restartable.
    model.save_pretrained(OUTPUT_DIR); tok.save_pretrained(OUTPUT_DIR)

In [ ]:
# Final held-out evaluation. These are the ONLY numbers that may be reported.
f1, truth, pred = evaluate(test_dl)
print('TEST macro_f1 = %.4f' % f1)
print()
if TASK == 'intent':
    present = sorted(set(truth.tolist()) | set(pred.tolist()))
    print(classification_report(truth, pred, labels=present,
          target_names=[LABELS[i] for i in present], zero_division=0))
else:
    print(classification_report(truth, pred, target_names=LABELS, zero_division=0))

print()
print('CAVEAT: measured on a held-out split of the scamshield SMS corpus.')
print('Not a call-transcript benchmark and not a VIVE end-to-end accuracy claim.')

In [ ]:
# Download the artifact, then place it under models/artifacts/ in the VIVE repo.
!cd /content && zip -qr vive-classifier.zip $(basename $OUTPUT_DIR) 2>/dev/null || true
from google.colab import files
files.download('/content/vive-classifier.zip')